# Quản lý Pipeline: Cleaning Quality AI
Notebook này giúp bạn quản lý luồng làm việc (workflow) và trực quan hóa từng bước trong quá trình chạy source code.

## Bước 1: Setup môi trường
> Trước khi chạy, hãy tạo file `.env` từ `.env.example` và điền các biến nhạy cảm (đặc biệt `ROBOFLOW_API_KEY`, `KAGGLE_USERNAME`, `KAGGLE_KEY`).
Cài thư viện cần thiết rồi kiểm tra trạng thái biến môi trường.

In [ ]:
import sys
import os
from pathlib import Path

# Co dinh thu muc lam viec ve goc project
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

def load_env(env_file='.env'):
    env_path = Path(env_file)
    if not env_path.exists():
        print('[WARN] Chua tim thay .env. Hay tao tu .env.example truoc khi tiep tuc.')
        return

    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

load_env()

req_path = os.path.abspath('requirements.txt')
print(f"[*] Thu muc hien tai: {os.getcwd()}")
print(f"[*] Duong dan cai dat: {req_path}")
print(f"[*] Dang dung Python tai: {sys.executable}")

for key_name in ['ROBOFLOW_API_KEY', 'KAGGLE_USERNAME', 'KAGGLE_KEY']:
    is_set = bool(os.getenv(key_name))
    print(f"[*] {key_name}: {'SET' if is_set else 'MISSING'}")

!{sys.executable} -m pip install -r "{req_path}"

[*] Thư mục hiện tại: e:\capstone\cleaning_ai_poc
[*] Đường dẫn cài đặt: e:\capstone\cleaning_ai_poc\requirements.txt
[*] Đang dùng Python tại: e:\capstone\cleaning_ai_poc\.venv\Scripts\python.exe
  Using cached kaggle-2.0.0-py3-none-any.whl.metadata (15 kB)
  Using cached datasets-4.8.4-py3-none-any.whl.metadata (19 kB)
  Using cached bleach-6.3.0-py3-none-any.whl.metadata (31 kB)
  Using cached python_slugify-8.0.4-py2.py3-none-any.whl.metadata (8.5 kB)
  Using cached pyarrow-23.0.1-cp311-cp311-win_amd64.whl.metadata (3.1 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached xxhash-3.6.0-cp311-cp311-win_amd64.whl.metadata (13 kB)
  Using cached multiprocess-0.70.19-py311-none-any.whl.metadata (7.5 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached huggingface_hub-1.8.0-py3-none-any.whl.metadata (13 kB)
  Using cached aiohttp-3.13.5-cp311-cp311-win_amd64.whl.metadat

## Bước 2: Tải và Chuẩn bị Dataset
Cell này chạy `src/download_dataset.py` theo cấu hình trong `.env`.
Nếu thiếu key, script sẽ cảnh báo rõ để tránh lộ secret trong source code.

In [ ]:
import os
from pathlib import Path

# Dam bao chay lenh o thu muc goc (root workspace)
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

print(f"[*] Thu muc hien tai: {os.getcwd()}")
print('[*] Dang khoi chay tai va chia split dataset theo .env...')

if not Path('.env').exists():
    print('[WARN] Khong co .env. Script van chay, nhung buoc can key co the bi bo qua.')

!{sys.executable} src/download_dataset.py

[*] Thư mục hiện tại: e:\capstone\cleaning_ai_poc
[*] Đang khởi chạy tải và chia split dataset...
CLEANING QUALITY AI — Dataset Downloader
[*] Đang tải Dirty vs Clean Room dataset từ Kaggle...
Dataset URL: https://www.kaggle.com/datasets/cdawn1/dirty-vs-clean-room
[ERROR] Không tải được qua API: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/DownloadDataset

[*] Dùng dataset thay thế...
[*] Thử tải từ Hugging Face...
[WARN] Không tải được HuggingFace: Dataset scripts are no longer supported, but found scene_parse_150.py
[ERROR] Quá trình tải dữ liệu thất bại. Không tạo ảnh synthetic theo yêu cầu.

[FAILED] Không thể lấy được dataset tự động. Quá trình chia cấu trúc folder bị hủy.
         Bạn cần tải file thủ công (vd: giải nén dataset vào 'data/raw/dirty_clean/').


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'scene_parse_150' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
e:\capstone\cleaning_ai_poc\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\phong\.cache\huggingface\hub\datasets--scene_parse_150. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as 

## Bước 3: Huấn luyện (Train) Mô Hình 
Bắt đầu train AI với tham số tuỳ chỉnh. Máy bạn dùng RTX 3050 (VRAM hạn chế), ta sẽ set `batch_size=16` và `epochs=10` để đánh giá. Nhìn quá trình chạy qua log bên dưới.

In [ ]:
# Lệnh Train
!{sys.executable} src/train.py --batch_size 16 --epochs 10

## Bước 4: Xem Biều đồ Curve Accuracy (Sau khi Train)
Kiểm tra xem mô hình của bạn có 'học' được bài toán hay không bằng biểu đồ. Mở ảnh `outputs/training_curves.png`.

In [ ]:
from IPython.display import Image, display

curve_path = "outputs/training_curves.png"
if os.path.exists(curve_path):
    print("Hiển thị biểu đồ Training:")
    display(Image(filename=curve_path))
else:
    print("Chưa có đồ thị. Vui lòng train xong mô hình ở cell trên.")

## Bước 5: Chạy Nghiệm Thu / Inference (Mô phỏng thực tế)
Cung cấp 1 cặp ảnh BEFORE / AFTER để tạo báo cáo Quality AI. Có thể dùng `--demo` để lấy ảnh ngẫu nhiên.

In [ ]:
# Chạy pipeline demo tạo file quality_report.png
!{sys.executable} src/infer.py --demo

# Hiển thị ảnh sau khi tạo ra trên Notebook
report_path = "outputs/quality_report.png"
if os.path.exists(report_path):
    print("\n[✓] QUẢN LÝ BÁO CÁO POCO:")
    display(Image(filename=report_path))
else:
    print("[!] Có lỗi lúc chạy Inference, chưa thấy tạo report.")